In [1]:
import pandas as pd
from building_models.utils.constants import (MAX_LENGTH_SEQUENCE, MIN_LENGTH_SEQUENCE,
                                             CANONICAL_RESIDUES)
from building_models.utils.utils_functions import UtilsFunctions

#### Antioxidant protein dataset filtering and quality control

This script applies quality control filters to the merged antioxidant protein dataset, evaluates sequence validity based on canonical residues and length constraints, and generates a filtered dataset along with metadata.

- Overview
    - Task: Antioxidant protein dataset filtering
    - Input: Merged dataset (CSV) and inconsistent sequences file (CSV)
    - Output: Filtered dataset (CSV), filtered inconsistent sequences (CSV), and metadata file (JSON)
- Process:
    - Load merged dataset and sequences with inconsistencies
    - Compute sequence lengths for all entries
    - Evaluate sequences based on canonical residue composition
    - Apply length constraints using predefined thresholds
    - Generate boolean filters for canonical validity and length range
    - Filter datasets based on combined quality criteria
    - Compute summary statistics for filtering steps
    - Export filtered datasets and metadata

- Auxiliary variables

In [2]:
path_to_export = "../../processed_dataset"

- Read data

In [3]:
df_data = pd.read_csv(f"{path_to_export}/merged_data/processed_dataset.csv")
df_sequences_with_errors = pd.read_csv(f"{path_to_export}/merged_data/sequences_with_erros.csv")

- Adding lengths to pending data

In [4]:
df_sequences_with_errors["length"] = df_sequences_with_errors["sequence"].str.len()

- Checking canonical residues

In [5]:
df_data["is_canon"] = df_data["sequence"].apply(UtilsFunctions.checking_canonical_residues, args=(False))
df_sequences_with_errors["is_canon"] = df_sequences_with_errors["sequence"].apply(UtilsFunctions.checking_canonical_residues, args=(False))

In [6]:
df_data["is_canon"].value_counts()

is_canon
True     7774
False      53
Name: count, dtype: int64

In [7]:
df_sequences_with_errors["is_canon"].value_counts()

is_canon
True    43
Name: count, dtype: int64

- Checking lengths

In [8]:
MIN_LENGTH_SEQUENCE, MAX_LENGTH_SEQUENCE

(2, 1024)

In [9]:
MIN_LENGTH_SEQUENCE = 70

In [10]:
df_data["is_in_length"] = df_data["length"].between(MIN_LENGTH_SEQUENCE, MAX_LENGTH_SEQUENCE)
df_data["is_in_length"].value_counts()

is_in_length
True     4221
False    3606
Name: count, dtype: int64

In [11]:
df_sequences_with_errors["is_in_length"] = df_sequences_with_errors["length"].between(MIN_LENGTH_SEQUENCE, MAX_LENGTH_SEQUENCE)
df_sequences_with_errors["is_in_length"].value_counts()

is_in_length
True     40
False     3
Name: count, dtype: int64

- Making filters

In [12]:
df_data_filtered = df_data[(df_data["is_canon"]) & (df_data["is_in_length"])]
df_sequences_with_errors_filtered = df_sequences_with_errors[(df_sequences_with_errors["is_canon"]) & (df_sequences_with_errors["is_in_length"])]

df_data_filtered.shape[0], df_sequences_with_errors_filtered.shape[0]

(4193, 40)

- Creating folder and exports

In [13]:
dict_metadata = {
    "total_data":{
        "processed_sequences" : df_data.shape[0],
        "sequences_with_errors" : df_sequences_with_errors.shape[0],
    },
    "canonical_sequences":{
        "used_vocab" : CANONICAL_RESIDUES,
        "processed_sequences":{
            "is_canon" : df_data[df_data["is_canon"]].shape[0],
            "is_not_canon" : df_data[df_data["is_canon"]==False].shape[0],
        },
        "sequences_with_error":{
            "is_canon" : df_sequences_with_errors[df_sequences_with_errors["is_canon"]].shape[0],
            "is_not_canon" : df_sequences_with_errors[df_sequences_with_errors["is_canon"]==False].shape[0],
        }
    },
    "length_evaluation":{
        "defined_length":{
            "min_value" : MIN_LENGTH_SEQUENCE,
            "max_value" : MAX_LENGTH_SEQUENCE,
        },
        "processed_sequences":{
            "is_in_length" : df_data[df_data["is_in_length"]].shape[0],
            "is_not_in_length" : df_data[df_data["is_in_length"]==False].shape[0],
        },
        "sequences_with_error" : {
            "is_in_length" : df_sequences_with_errors[df_sequences_with_errors["is_in_length"]].shape[0],
            "is_not_in_length" : df_sequences_with_errors[df_sequences_with_errors["is_in_length"] == False].shape[0],
        }
    },
    "final_evaluation":{
        "processed_sequences" : df_data_filtered.shape[0],
        "positive_examples" : df_data_filtered[df_data_filtered["label"] == 1].shape[0],
        "negative_examples" : df_data_filtered[df_data_filtered["label"] == 0].shape[0],
        "sequences_with_errors" : df_sequences_with_errors_filtered.shape[0],
    }
}

dict_metadata

{'total_data': {'processed_sequences': 7827, 'sequences_with_errors': 43},
 'canonical_sequences': {'used_vocab': ['A',
   'C',
   'D',
   'E',
   'F',
   'G',
   'H',
   'I',
   'K',
   'L',
   'M',
   'N',
   'P',
   'Q',
   'R',
   'S',
   'T',
   'V',
   'W',
   'Y'],
  'processed_sequences': {'is_canon': 7774, 'is_not_canon': 53},
  'sequences_with_error': {'is_canon': 43, 'is_not_canon': 0}},
 'length_evaluation': {'defined_length': {'min_value': 70, 'max_value': 1024},
  'processed_sequences': {'is_in_length': 4221, 'is_not_in_length': 3606},
  'sequences_with_error': {'is_in_length': 40, 'is_not_in_length': 3}},
 'final_evaluation': {'processed_sequences': 4193,
  'positive_examples': 1010,
  'negative_examples': 3183,
  'sequences_with_errors': 40}}

- Exporting data

In [14]:
UtilsFunctions.make_directory(f"{path_to_export}/processed_data")
UtilsFunctions.export_json(f"{path_to_export}/processed_data/metadata_process.json", dict_metadata)
df_data_filtered.to_csv(f"{path_to_export}/processed_data/processed_dataset.csv", index=False)
df_sequences_with_errors_filtered.to_csv(f"{path_to_export}/processed_data/sequences_with_errors.csv", index=False)